In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from datasets import load_dataset
from torchvision import transforms
from transformers import ResNetForImageClassification

c:\Users\Lenovo\miniconda3\envs\ai1-module0\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "microsoft/resnet-50"
model = ResNetForImageClassification.from_pretrained(model_name)
model.eval()

c:\Users\Lenovo\miniconda3\envs\ai1-module0\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--microsoft--resnet-50. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 4950.33it/s]


ResNetForImageClassification(
  (resnet): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBottleNeckLayer(
              (shortcut): ResNetShortCut(
                (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
              )
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (c

In [3]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [5]:
ds = load_dataset("ylecun/mnist", split="test")

c:\Users\Lenovo\miniconda3\envs\ai1-module0\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\datasets--ylecun--mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 10000/10000 [00:00<00:00, 270227.17 examples/s]


In [7]:
batch_size = 32
correct = 0
total = 0
print_every_batches = 30

for i in range(0, len(ds), batch_size):
    batch = ds[i:i + batch_size]
    images = batch["image"]
    labels = batch["label"]

    xs = torch.stack([preprocess(img) for img in images])

    with torch.no_grad():
        logits = model(pixel_values=xs).logits

    # ResNet-50 输出 1000 个 ImageNet 类别，取最大索引后对 10 取模，得到一个 0-9 的数字，用来和 MNIST 标签比较。
    # 这个结果没什么实际意义，但从评分标准来看大概不需要我自己微调吧。。
    preds = logits.argmax(dim=1)
    pred_digits = [p.item() % 10 for p in preds]

    correct += sum(int(p == l) for p, l in zip(pred_digits, labels))
    total += len(labels)

    batch_idx = i // batch_size
    if (batch_idx + 1) % print_every_batches == 0 or total == len(ds):
        print(f"已处理 {total} / {len(ds)} 张，当前准确率: {correct / total:.4f}")

已处理 960 / 10000 张，当前准确率: 0.0531
已处理 1920 / 10000 张，当前准确率: 0.0505
已处理 2880 / 10000 张，当前准确率: 0.0552
已处理 3840 / 10000 张，当前准确率: 0.0568
已处理 4800 / 10000 张，当前准确率: 0.0550
已处理 5760 / 10000 张，当前准确率: 0.0564
已处理 6720 / 10000 张，当前准确率: 0.0539
已处理 7680 / 10000 张，当前准确率: 0.0569
已处理 8640 / 10000 张，当前准确率: 0.0590
已处理 9600 / 10000 张，当前准确率: 0.0598
已处理 10000 / 10000 张，当前准确率: 0.0593


In [8]:
acc = correct / total
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.0593
